# Reading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
building_structure = pd.read_csv('csv_building_structure.csv')
building_ownership = pd.read_csv('csv_building_ownership_and_use.csv')

# Data Cleaning

Creating 1 table from both tables with required features

In [ ]:
len(building_structure.columns)

In [ ]:
building_structure.drop(['condition_post_eq', 'technical_solution_proposed'], axis = 1, inplace = True)

In [ ]:
building_structure.columns.values

In [ ]:
building_structure.drop(['count_floors_post_eq', 'height_ft_post_eq'],axis = 1, inplace = True)

In [ ]:
len(building_structure.columns.values)

In [ ]:
building_structure.shape

In [ ]:
building_ownership.columns.values

In [ ]:
building_ownership.shape

In [ ]:
building_ownership.drop(['district_id', 'vdcmun_id', 'ward_id'], axis = 1 , inplace = True)

In [ ]:
building_ownership.shape

In [ ]:
e_data = pd.merge(building_structure,building_ownership, on = 'building_id')

In [ ]:
e_data.head()

In [ ]:
e_data.shape

Checking for the null values in the table and their treatment

In [ ]:
e_data.isna().sum()

Because damage grade is what we need to predict damage and NULL values in target variable i.e damage grade cant be imputed  and because the NULL values are really less so we can drop those rows from the table

In [ ]:
e_data = e_data.dropna(subset = ['damage_grade'],axis= 0)

In [ ]:
e_data.isna().sum()

In [ ]:
e_data.columns.values

In [ ]:
e_data.head()

After dropping the data we have only 1 NULL value in one of the columns so it implies that the rows with empty target variables had NULL in other columns also so the data data collection was not done properly for those houses 

We will treat the missing value in count_families column while treating the outliers


# Outliers and its treatment

In [ ]:
#for count_floor_pre_eq
e_data['count_floors_pre_eq'].describe()

In [ ]:
e_data['count_floors_pre_eq'].value_counts()

In [ ]:
e_data[e_data['count_floors_pre_eq']<2]['count_floors_pre_eq'].count()

In [ ]:
e_data[e_data['count_floors_pre_eq']>2]['count_floors_pre_eq'].count()

In [ ]:
e_data[e_data['count_floors_pre_eq']==2]['count_floors_pre_eq'].count()

Because the data has maximum of 9 floors only so this can be treates as ordinal datatype so no need of outlier removal and visualization

In [ ]:
#for age_building

In [ ]:
e_data['age_building'].describe()

In [ ]:
e_data[e_data['age_building']==999]['age_building'].count()

In [ ]:
sns.distplot(e_data['age_building'])

In [ ]:
pd.crosstab(e_data[e_data['age_building']>100]['age_building'], e_data['damage_grade'])

In [ ]:
e_data[e_data['age_building']>100]['damage_grade'].value_counts() 

Because we need only 3 damage categories so we can cap grade 4 and grade 5 to grade 3 in damage_grade column which is our target variable

In [ ]:
e_data['damage_grade'] = e_data['damage_grade'].str.replace('Grade','')

In [ ]:
e_data['damage_grade'] = e_data['damage_grade'].astype(float).astype(int)
e_data['damage_grade'] = e_data['damage_grade'].apply(lambda x:3 if(x>3) else x)


In [ ]:
e_data['damage_grade'].head()

In [ ]:
pd.crosstab(e_data[e_data['age_building']>54]['age_building'],e_data['damage_grade'])

In [ ]:
#e_data[e_data['age_building']>54]['damage_grade'].value_counts()

In [ ]:
sns.boxplot(e_data['age_building'])

Because there are outliers in the age_building so we will use IQR method(By using Q3+1.5*IQR we get upper whisker = 54) to treat outlier and also the data with age_building >54 have more Grade 3 damage and it is approximately same for al the data so we can cap the age to 54 

In [ ]:
#e_data['age_building'] = e_data['age_building'].apply(lambda x:54 if (x>54) else x)

In [ ]:
sns.kdeplot(e_data['age_building'])

In [ ]:
#for plinth_area_sq_ft

In [ ]:
e_data['plinth_area_sq_ft'].describe()

In [ ]:
sns.boxplot(e_data['plinth_area_sq_ft'], orient = 'v')

In [ ]:
upper_whisker = e_data['plinth_area_sq_ft'].quantile(0.75)+1.5*(e_data['plinth_area_sq_ft'].quantile(0.75)-e_data['plinth_area_sq_ft'].quantile(0.25))
upper_whisker

In [ ]:
e_data[e_data['plinth_area_sq_ft']>e_data['plinth_area_sq_ft'].quantile(0.75)]['damage_grade'].value_counts()

In [ ]:
e_data[e_data['plinth_area_sq_ft']>upper_whisker]['damage_grade'].value_counts()

Because the data shows same behaviour for the data above upper whisker in plinth_area_sq_ft  so we are capping the outliers to upper whisker

In [ ]:
e_data['plinth_area_sq_ft'] = e_data['plinth_area_sq_ft'].apply(lambda x:755 if (x>755) else x)

In [ ]:
sns.kdeplot(e_data['plinth_area_sq_ft'])

In [ ]:
sns.boxplot(e_data['plinth_area_sq_ft'])

In [ ]:
#for height_ft_pre_eq

In [ ]:
e_data['height_ft_pre_eq'].describe()

Because the number of number of floor is dependent on height of building so lets check the relationship through a boxplot 

In [ ]:
sns.boxplot(e_data['count_floors_pre_eq'],e_data['height_ft_pre_eq'])

No meaningful data is conveyed by the boxplot so we will check the height_ft_pre_eq column for further analysis

In [ ]:
f, ax = plt.subplots(figsize=(12,9))
sns.boxplot(e_data['damage_grade'],e_data['height_ft_pre_eq'])

from the above plot we can see that height of the building doesnot show any significant spread hence the relationship is not much with damage_grade

In [ ]:
e_data[['damage_grade','height_ft_pre_eq']].corr()

Now converting the categorical columns in the required format

In [ ]:
#for land_surface_condition
e_data['land_surface_condition'].value_counts()

In [ ]:
def value_change(x):
    if 'Flat' in x:
        return (x.replace('Flat', 'n'))   
    if 'Moderate slope' in x:
        return (x.replace('Moderate slope', 'o'))
    if 'Steep slope' in x:
        return (x.replace('Steep slope', 't'))

In [ ]:
e_data['land_surface_condition']=e_data['land_surface_condition'].apply(value_change)
e_data['land_surface_condition'].value_counts()

In [ ]:
#for foundation_type
e_data['foundation_type'].value_counts()

In [ ]:
def value_change1(x):
    if 'Mud mortar-Stone/Brick' in x:
        return (x.replace('Mud mortar-Stone/Brick', 'h'))   
    if 'Bamboo/Timber' in x:
        return (x.replace('Bamboo/Timber', 'i'))
    if 'Cement-Stone/Brick' in x:
        return (x.replace('Cement-Stone/Brick', 'r'))
    if 'RC' in x:
        return (x.replace('RC', 'u'))
    if 'Other' in x:
        return (x.replace('Other', 'w'))

In [ ]:
e_data['foundation_type']=e_data['foundation_type'].apply(value_change1)
e_data['foundation_type'].value_counts()

In [ ]:
#for roof_type
e_data['roof_type'].value_counts()

In [ ]:
def value_change2(x):
    if 'Bamboo/Timber-Light roof' in x:
        return (x.replace('Bamboo/Timber-Light roof', 'n'))   
    if 'Bamboo/Timber-Heavy roof' in x:
        return (x.replace('Bamboo/Timber-Heavy roof', 'q'))
    if 'RCC/RB/RBC' in x:
        return (x.replace('RCC/RB/RBC', 'x'))
   

In [ ]:
e_data['roof_type']=e_data['roof_type'].apply(value_change2)
e_data['roof_type'].value_counts()

In [ ]:
# for ground_floor_type
e_data['ground_floor_type'].value_counts()

In [ ]:
def value_change3(x):
    if 'Mud' in x:
        return (x.replace('Mud', 'f'))   
    if 'RC' in x:
        return (x.replace('RC', 'm'))
    if 'Brick/Stone' in x:
        return (x.replace('Brick/Stone', 'v'))
    if 'Timber' in x:
        return (x.replace('Timber', 'x'))
    if 'Other' in x:
        return (x.replace('Other', 'z'))

In [ ]:
e_data['ground_floor_type']=e_data['ground_floor_type'].apply(value_change3)
e_data['ground_floor_type'].value_counts()

In [ ]:
# for other_floor_type
e_data['other_floor_type'].value_counts()


In [ ]:
def value_change4(x):
    if 'TImber/Bamboo-Mud' in x:
        return (x.replace('TImber/Bamboo-Mud', 'j'))   
    if 'Timber-Planck' in x:
        return (x.replace('Timber-Planck', 'q'))
    if 'Not applicable' in x:
        return (x.replace('Not applicable', 's'))
    if 'RCC/RB/RBC' in x:
        return (x.replace('RCC/RB/RBC', 'x'))
   

In [ ]:
e_data['other_floor_type']=e_data['other_floor_type'].apply(value_change4)
e_data['other_floor_type'].value_counts()

In [ ]:
# for position
e_data['position'].value_counts()

In [ ]:
def value_change5(x):
    if 'Not attached' in x:
        return (x.replace('Not attached', 'j'))   
    if 'Attached-1 side' in x:
        return (x.replace('Attached-1 side', 'o'))
    if 'Attached-2 side' in x:
        return (x.replace('Attached-2 side', 's'))
    if 'Attached-3 side' in x:
        return (x.replace('Attached-3 side', 't'))

In [ ]:
e_data['position']=e_data['position'].apply(value_change5)
e_data['position'].value_counts()

In [ ]:
# for plan_configuration
e_data['plan_configuration'].value_counts()

In [ ]:
def value_change6(x):
    if 'Rectangular' in x:
        return (x.replace('Rectangular', 'a'))   
    if 'Square' in x:
        return (x.replace('Square', 'c'))
    if 'L-shape' in x:
        return (x.replace('L-shape', 'd'))
    if 'T-shape' in x:
        return (x.replace('T-shape', 'f'))
    if 'Multi-projected' in x:
        return (x.replace('Multi-projected', 'm'))   
    if 'Others' in x:
        return (x.replace('Others', 'n'))
    if 'U-shape' in x:
        return (x.replace('U-shape', 'o'))
    if 'E-shape' in x:
        return (x.replace('E-shape', 'q'))
    if 'Building with Central Courtyard' in x:
        return (x.replace('Building with Central Courtyard', 's'))
    if 'H-shape' in x:
        return (x.replace('H-shape', 'u'))

In [ ]:
e_data['plan_configuration'] = e_data['plan_configuration'].apply(value_change6)
e_data['plan_configuration'].value_counts()

In [ ]:
# for legal_ownership_status
e_data['legal_ownership_status'].value_counts()

In [ ]:
def value_change7(x):
    if 'Private' in x:
        return (x.replace('Private', 'a'))   
    if 'Public' in x:
        return (x.replace('Public', 'r'))
    if 'Institutional' in x:
        return (x.replace('Institutional', 'v'))
    if 'Other' in x:
        return (x.replace('Other', 'w'))

In [ ]:
e_data['legal_ownership_status'] = e_data['legal_ownership_status'].apply(value_change7)
e_data['legal_ownership_status'].value_counts()

In [ ]:
# for count_families
e_data['count_families'].describe()

In [ ]:
e_data['count_families'].value_counts()

Because there is 1 missing value so we will impute the column with the maximum occuring value which is 1 

In [ ]:
e_data['count_families'].fillna(1, inplace = True )
e_data['count_families']= e_data['count_families'].astype(int)


# Data Visualization

Lets check the various relationship between different columns of the table 

In [ ]:
# plinth_area_sq_ft and count_floor_pre_eq

sns.boxplot(e_data['count_floors_pre_eq'], e_data['plinth_area_sq_ft'])

The box plot shows that on an average as the number of floor increases the average  plinth_area_size also increases

In [ ]:
cat_cols = list(e_data.select_dtypes(include = ['object']).columns)
dfc = pd.get_dummies(e_data,columns = cat_cols, drop_first = True)


In [ ]:
dfc['damage_grade'].value_counts()

Lets cluster the data to find the grades and see how the machine is clustering the data

In [ ]:
from scipy.stats import zscore

In [ ]:
dfc[['age_building','plinth_area_sq_ft', 'height_ft_pre_eq']] = dfc[['age_building','plinth_area_sq_ft', 'height_ft_pre_eq']].apply(zscore)

In [ ]:
dfc[['age_building','plinth_area_sq_ft', 'height_ft_pre_eq']].head()

Separating Gorkha district data which will be our test data and rest other district will serve as our test data to predict the damage grade

In [ ]:
dfc[dfc['district_id']==36].to_csv('earthquake_test.csv')

In [ ]:
dfc[dfc['district_id']!=36].to_csv('earthquake_train.csv')

In [ ]:
dfc.head()

In [ ]:
X.drop('label', axis =1, inplace = True)

In [ ]:
dfc['damage_grade'].value_counts()

In [ ]:
X.head()

With KMeans we cant get desireable output and agglomerative clustering is not working on such a large dataset let us check with other methods

In [ ]:
sns.countplot(e_data['ground_floor_type'], hue = e_data['damage_grade'])

In [ ]:
f, ax = plt.subplots(figsize=(12,9))
sns.countplot(e_data['land_surface_condition'], hue = e_data['damage_grade'])

In [ ]:
f, ax = plt.subplots(figsize=(12,9))
sns.countplot(e_data['land_surface_condition'], hue = e_data['count_floors_pre_eq'])

In [ ]:
f, ax = plt.subplots(figsize=(12,9))
sns.countplot(e_data['count_families'], hue = e_data['damage_grade'])

In [ ]:
sns.countplot(e_data['other_floor_type'], hue = e_data['damage_grade'])

In [ ]:
sns.countplot(e_data['other_floor_type'], hue = e_data['ground_floor_type'])

In [ ]:
e_data.info()

In [ ]:
e_data.shape

In [ ]:
e_data_train = e_data[e_data['district_id']!=36]

In [ ]:
e_data_test = e_data[e_data['district_id']==36]

In [ ]:
X_train = e_data.drop(['building_id', 'district_id', 'vdcmun_id', 'ward_id','damage_grade'], axis = 1)
y_train = e_data['damage_grade']

In [ ]:
e_data_train.to_csv('e_data_train.csv')
e_data_test.to_csv('e_data_test.csv')

In [ ]:
e_data_test['damage_grade'].value_counts()